In [ ]:
import pandas as pd

import os
import s3fs


fs = s3fs.S3FileSystem(
    client_kwargs={'endpoint_url': 'https://'+'minio.lab.sspcloud.fr'},
    key = os.environ["AWS_ACCESS_KEY_ID"], 
    secret = os.environ["AWS_SECRET_ACCESS_KEY"], 
    token = os.environ["AWS_SESSION_TOKEN"])


with fs.open("s3://lab/art_net_dec.parquet") as f : 
    dfa = pd.read_parquet(f)
"""
with fs.open("s3://lab/conf_net_dec.parquet") as f : 
    dfc = pd.read_parquet(f)"""

#with fs.open("s3://lab/mem/full_name_list.csv") as f : 
   # nl = pd.read_csv(f, index_col=[0])

with fs.open("s3://lab/mem/n_to_g.csv") as f : 
    ntg = pd.read_csv(f, index_col=[0])
"""
with fs.open("s3://lab/kgnd.csv") as f:
    kgnd = pd.read_csv(f)
with fs.open("s3://lab/ignd.csv") as f:
    ignd = pd.read_csv(f)
with fs.open("s3://lab/wgnd.csv") as f:
    wgnd = pd.read_csv(f)
with fs.open("s3://lab/jgnd.csv") as f:
    jgnd = pd.read_csv(f)
with fs.open("s3://lab/cgnd.csv") as f:
    cgnd = pd.read_csv(f)
with fs.open("s3://lab/usgnd.csv") as f: 
    usgnd = pd.read_csv(f)"""

'\nwith fs.open("s3://lab/kgnd.csv") as f:\n    kgnd = pd.read_csv(f)\nwith fs.open("s3://lab/ignd.csv") as f:\n    ignd = pd.read_csv(f)\nwith fs.open("s3://lab/wgnd.csv") as f:\n    wgnd = pd.read_csv(f)\nwith fs.open("s3://lab/jgnd.csv") as f:\n    jgnd = pd.read_csv(f)\nwith fs.open("s3://lab/cgnd.csv") as f:\n    cgnd = pd.read_csv(f)\nwith fs.open("s3://lab/usgnd.csv") as f: \n    usgnd = pd.read_csv(f)'

In [ ]:
gnd=pd.concat([wgnd,cgnd,kgnd,ignd,jgnd,usgnd],axis=0,ignore_index=True)
gnd=gnd.drop(columns=["Count"])
gnd["Name"] = gnd["Name"].str.lower()
gnd = gnd.drop_duplicates()
gnd

,Name,Gender
0,baby,F
1,aisyah,F
2,anela,F
3,anela,F
4,fiyinfoluwa,F
...,...,...
27351475,Zylyn,M
27351476,Zymiere,M
27351477,Zypher,M
27351478,Zyre,M


In [22]:
#duplicates
pd.concat(g for _, g in gnd.groupby("Name") if len(g) > 1)

,Name,Gender
15,a,M
17,a,F
686,aaban,M
26788664,aaban,M
1272,aabha,F
...,...,...
26860529,zyvion,M
21168275,zyyanna,F
26877519,zyyanna,F
21168350,zzyzx,M


In [8]:
dfa["authors"] = dfa["authors"].apply(lambda lst: [d["name"] for d in lst])
dfc["authors"] = dfc["authors"].apply(lambda lst: [d["name"] for d in lst])

In [16]:
nla = dfa["authors"].explode().str.split(' ').str[0]
nlc = dfc["authors"].explode().str.split(' ').str[0]

In [19]:
nl = pd.concat([nla,nlc]).drop_duplicates()

In [17]:
nl = nl[~nl["authors"].str.contains(r"^[A-Za-z]\.$", na=False)]
nl = nl[~nl["authors"].str.contains(r"^[A-Za-z]\.-[A-Za-z]\.$", na=False)]
nl = nl[~nl["authors"].str.contains(r"^[A-Za-z]-[A-Za-z]\.$", na=False)]
nl = nl[~nl["authors"].str.contains(r"^[A-Za-z][A-Za-z]\.$", na=False)]
nl = nl[~nl["authors"].str.contains(r"^[A-Za-z][A-Za-z][A-Za-z]\.$", na=False)]

In [51]:
nl.to_csv("s3://lab/mem/full_name_list.csv")

In [18]:
nl = nl["authors"].str.replace("-", " ").str.split(" ").str[0].str.lower()

In [28]:
pd.DataFrame(nl)

,authors
0,kai
1,richard
1,stefan
2,herbert
3,sophie
...,...
3749381,siwaphon
3749388,tasi
3749413,xiangxu
3749415,pradhyumansinh


In [ ]:
ntg = pd.DataFrame(nl).merge(gnd,how="left",left_on="authors",right_on="Name")

In [43]:
ntg[~ntg["Gender"].isna()]

,authors,Name,Gender
0,kai,kai,M
1,richard,richard,M
2,stefan,stefan,M
3,herbert,herbert,M
4,sophie,sophie,F
...,...,...,...
313617,chien,chien,M
313625,chiu,chiu,M
313628,tasi,tasi,M
313629,xiangxu,xiangxu,M


In [37]:
gnd = gnd.drop_duplicates(subset="Name")

In [53]:
!pip install gender-guesser

In [71]:
import gender_guesser.detector as gg

d = gg.Detector()
ng_dict = pd.DataFrame()
ng_dict["name"] = nl
ng_dict["gender"] = nl.apply(lambda x: d.get_gender(x))
ng_dict

,name,gender
0,Kai,male
1,Richard,male
1,Stefan,male
2,Herbert,male
3,Sophie,female
...,...,...
3749381,Siwaphon,unknown
3749388,Tasi-Ying,unknown
3749413,XiangXu,unknown
3749415,Pradhyumansinh,unknown


In [72]:
ng_dict[ng_dict["gender"]=="unknown"]

,name,gender
4,NaN,unknown
27,Hans-Jürgen,unknown
42,Nicolaos,unknown
87,Brikenrikena,unknown
96,Catalin,unknown
...,...,...
3749381,Siwaphon,unknown
3749388,Tasi-Ying,unknown
3749413,XiangXu,unknown
3749415,Pradhyumansinh,unknown


In [44]:
ntg.to_csv("s3://lab/mem/n_to_g.csv")

In [ ]:
dfa["authors"] = dfa["authors"].apply(lambda lst: [d["name"] for d in lst])
attr = dfa.explode("authors")
attr["name"] = attr["authors"].str.split(" ").str[0].str.lower()
attr = attr.merge(ntg[["authors","Gender"]],how="left",left_on="name",right_on="authors")
attr